# Day 04: Ops:Byte Ratio & Arithmetic Intensity
> *Inference Engineering* — Chapter 2.4 | Philip Kiely, Baseten Books 2026

**Layer:** Runtime | **Prerequisite:** Day 03 (MoE)


## Concept Overview

Arithmetic intensity (ops:byte ratio) is the number of floating point operations performed per byte of memory transferred. It determines whether a kernel is compute-bound or memory-bound. The roofline model plots achievable FLOP/s as a function of arithmetic intensity, with two ceilings: the memory bandwidth ceiling and the compute ceiling. For LLM inference, the decode phase (batch=1) is deeply memory-bound — loading model weights dominates. Batching increases arithmetic intensity by amortizing weight loads across multiple tokens.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    props = torch.cuda.get_device_properties(0)
    print(f'Memory: {props.total_memory/1e9:.1f} GB')


## 1. The Roofline Model

For a kernel with arithmetic intensity $I$ (FLOP/byte):

$$\text{Attainable FLOP/s} = \min(\text{Peak FLOP/s},\ I \times \text{Memory BW})$$

The ridge point $I^* = \text{Peak FLOP/s} / \text{Memory BW}$ separates memory-bound (left) from compute-bound (right).


In [ ]:
# GPU specs: A100 80GB SXM
gpu_specs = {
    'A100 80GB': {'tflops_fp16': 312, 'mem_bw_tbs': 2.0},   # 2 TB/s HBM2e
    'H100 SXM':  {'tflops_fp16': 989, 'mem_bw_tbs': 3.35},  # 3.35 TB/s HBM3
    'DGX Spark': {'tflops_fp16': 67,  'mem_bw_tbs': 0.273}, # GB10 ~273 GB/s
}

fig, ax = plt.subplots(figsize=(10, 6))
intensities = np.logspace(-1, 3, 500)  # 0.1 to 1000 FLOP/byte

colors = ['blue', 'orange', 'green']
for (name, specs), color in zip(gpu_specs.items(), colors):
    peak = specs['tflops_fp16'] * 1e12  # FLOP/s
    bw = specs['mem_bw_tbs'] * 1e12    # byte/s
    ridge = peak / bw
    attainable = np.minimum(peak, intensities * bw)
    ax.loglog(intensities, attainable / 1e12, label=f'{name} (ridge={ridge:.0f} FLOP/B)', color=color)
    ax.axvline(ridge, color=color, linestyle=':', alpha=0.5)

# Mark typical LLM operations
ops = {
    'Decode (bs=1)': 1,
    'Decode (bs=32)': 32,
    'Prefill (seq=512)': 512,
    'GEMM (large)': 1000,
}
for label, intensity in ops.items():
    ax.axvline(intensity, color='red', linestyle='--', alpha=0.3)
    ax.text(intensity * 1.1, 1, label, rotation=90, fontsize=8, color='red')

ax.set_xlabel('Arithmetic Intensity (FLOP/byte)')
ax.set_ylabel('Attainable FLOP/s (TFLOP/s)')
ax.set_title('Roofline Model — LLM Inference Regimes')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.savefig('roofline_model.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved roofline_model.png')


## 2. Arithmetic Intensity of LLM Operations

A matrix multiply $y = Wx$ where $W \in \mathbb{R}^{m \times k}$:
- **FLOPs:** $2mk$ (multiply-accumulate)
- **Bytes read:** $(mk + k) \times \text{dtype\_size}$
- **Arithmetic intensity:** $\approx 2mk / (mk \cdot \text{dtype\_size}) = 2/\text{dtype\_size}$ for decode (batch=1)

For batch size $B$: intensity $\approx 2B/\text{dtype\_size}$. This is why batching is the primary lever for moving from memory-bound to compute-bound.


In [ ]:
def matmul_intensity(m, k, batch, dtype_bytes=2):
    """Arithmetic intensity for batched matmul y=xW, x:[batch,k], W:[k,m]"""
    flops = 2 * batch * m * k
    # Weight W dominates reads when batch is small
    bytes_read = (m * k + batch * k + batch * m) * dtype_bytes
    return flops / bytes_read

# Llama-3-8B layer: d_model=4096, d_ff=14336
d_model, d_ff = 4096, 14336
print('Arithmetic Intensity for Llama-3-8B FFN layer (W:[d_ff, d_model])')
print(f'{"Batch Size":>12} {"Intensity (FLOP/B)":>20} {"Regime":>15}')
print('-' * 50)
ridge_a100 = 312e12 / 2e12  # 156 FLOP/byte for A100
for bs in [1, 2, 4, 8, 16, 32, 64, 128, 256]:
    intensity = matmul_intensity(d_ff, d_model, bs)
    regime = 'COMPUTE-BOUND' if intensity > ridge_a100 else 'memory-bound'
    print(f'{bs:>12} {intensity:>20.1f} {regime:>15}')


## 3. Measuring Real Memory Bandwidth

We can empirically measure how close to peak bandwidth a simple memory-bound operation achieves.


In [ ]:
import time

def measure_bandwidth(size_mb=1024, dtype=torch.float16, device='cpu'):
    """Measure achievable memory bandwidth via a vector copy."""
    n = size_mb * 1024 * 1024 // 2  # FP16 = 2 bytes
    a = torch.randn(n, dtype=dtype, device=device)
    b = torch.empty_like(a)

    # Warmup
    for _ in range(3):
        b.copy_(a)
    if device != 'cpu':
        torch.cuda.synchronize()

    t0 = time.perf_counter()
    iters = 10
    for _ in range(iters):
        b.copy_(a)
    if device != 'cpu':
        torch.cuda.synchronize()
    t1 = time.perf_counter()

    bytes_moved = 2 * n * 2 * iters  # read + write, FP16
    bw_gb_s = bytes_moved / (t1 - t0) / 1e9
    return bw_gb_s

cpu_bw = measure_bandwidth(256, device='cpu')
print(f'CPU memory bandwidth: {cpu_bw:.1f} GB/s')

if torch.cuda.is_available():
    gpu_bw = measure_bandwidth(1024, device='cuda')
    print(f'GPU memory bandwidth: {gpu_bw:.1f} GB/s')
    props = torch.cuda.get_device_properties(0)
    print(f'GPU peak BW (spec): ~273 GB/s for DGX Spark GB10')
    print(f'Efficiency: {gpu_bw/273*100:.0f}% of peak')


## Experiments: Try These

1. **Measure GEMM efficiency**: Use `torch.utils.benchmark` to measure FLOP/s for matrix multiplies at different sizes. Compare against theoretical peak.
2. **Batch size sweep**: Run a decode-style matmul (tall, skinny W) across batch sizes 1-256 and plot latency vs throughput — find the knee of the curve.
3. **Mixed precision**: Compare arithmetic intensity for FP32 vs FP16 vs INT8. How does quantization affect the memory-bound threshold?


## Key Takeaways

- Arithmetic intensity = FLOPs / bytes transferred; it determines whether a kernel is compute-bound or memory-bound.
- The roofline model shows attainable FLOP/s = min(peak compute, intensity × memory BW).
- LLM decode with batch=1 has intensity ~1 FLOP/byte — deeply memory-bound; the GPU spends most time waiting on HBM.
- Increasing batch size linearly increases arithmetic intensity — the primary mechanism for improving GPU utilization during inference.

## References
- *Inference Engineering* Ch 2.4 — Philip Kiely, Baseten Books 2026
- Williams et al. (2009), "Roofline: An Insightful Visual Performance Model"
- NVIDIA A100 Datasheet, NVIDIA H100 Datasheet
